In [ ]:
class CustomLinearRegression:
    def __init__(self, learning_rate=0.01, iterations=1000):
        self.lr = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = None
        self.cost_history = []

    def fit(self, X, y):
        num_samples, num_features = X.shape
        self.weights = np.zeros(num_features)
        self.bias = 0
        
        for i in range(self.iterations):
            # Hypothesis calculation
            y_predicted = np.dot(X, self.weights) + self.bias
            
            # Compute gradients across the entire batch
            dw = (1 / num_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / num_samples) * np.sum(y_predicted - y)
            
            # Update mathematical parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            # Record cost function (MSE)
            cost = (1 / (2 * num_samples)) * np.sum((y_predicted - y) ** 2)
            self.cost_history.append(cost)
            
            if i % 100 == 0:
                print(f"Iteration {i}: Cost = {cost:.4f}")

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

# Initialize and train custom mathematical model
# Note: y_train requires conversion to a numpy array for matrix operations to process correctly
custom_model = CustomLinearRegression(learning_rate=0.1, iterations=1500)
custom_model.fit(X_train_processed, y_train.values)

# Predict on test set
custom_preds = custom_model.predict(X_test_processed)

In [ ]:
from sklearn.linear_model import LinearRegression

# Train Scikit-Learn Baseline Model
sklearn_model = LinearRegression()
sklearn_model.fit(X_train_processed, y_train)
sklearn_preds = sklearn_model.predict(X_test_processed)

# Evaluation Script for Metrics Comparison
def evaluate_model(y_true, y_pred, model_name):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"--- {model_name} Evaluation ---")
    print(f"R2 Score: {r2:.4f} (Target: > 0.83)")
    print(f"RMSE:     {rmse:.4f} (Target: < 0.39)")
    print("-" * 30)
    return r2, rmse

print("\nEvaluating Comparative Metrics:")
eval_custom = evaluate_model(y_test, custom_preds, "Custom Gradient Descent")
eval_sklearn = evaluate_model(y_test, sklearn_preds, "Scikit-Learn Baseline")

# Visualizing Cost Reduction over iterations for empirical mathematical proof
plt.plot(custom_model.cost_history)
plt.title('Cost Function over Gradient Descent Iterations')
plt.xlabel('Iterations')
plt.ylabel('Cost (MSE)')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Dataset
# Restricting data exclusively to the designated training set as mandated.
df = pd.read_csv('UNSW-NB15-training-set.csv')

# 2. Data Cleaning & Column Selection
# Dropping identifiers and ground truth classification columns to prevent data leakage.
# The classification columns in UNSW-NB15 are 'label' and 'attack_cat', and 'id' is the identifier.
cols_to_drop = ['id', 'label', 'attack_cat']
df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)

# 3. Exploratory Data Analysis (EDA)
# Visualizing the distribution of the target variable to understand the baseline shape.
plt.figure(figsize=(10, 6))
sns.histplot(df['dur'], bins=50, kde=True)
plt.title('Distribution of Network Flow Duration (dur) Before Transformation')
plt.xlabel('Duration (seconds)')
plt.ylabel('Frequency')
plt.show()

# 4. Skewness Transformation
# Identifying features with extreme long-tail spread (skewness > 2) and applying logarithmic transformations.
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns
skewed_feats = df[numeric_features].apply(lambda x: x.skew()).sort_values(ascending=False)

# Filter for skewness > 2
high_skew = skewed_feats[skewed_feats > 2].index
print(f"Features requiring log transformation (skew > 2): {list(high_skew)}")

# Apply log1p transformation to handle potential zero values in network data safely.
for feature in high_skew:
    df[feature] = np.log1p(df[feature])

# The continuous target variable 'dur' must also be transformed according to task requirements.
if 'dur' not in high_skew:
    df['dur'] = np.log1p(df['dur'])

# 5. Train-Test Split (80/20)
# Performing the required 80/20 train-test split before scaling to prevent data snooping.
X = df.drop(columns=['dur'])
y = df['dur']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 6. Missing Data Handling, Encoding & Scaling
# Implementing appropriate imputation for continuous and categorical features.
# Applying OneHot Encoding for categorical features and Standard Scaling for continuous features.
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Fit on training data and transform both train and test sets to construct the final pipelines
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)